# 03 Models: Logistic Regression, Naive Bayes, Decision Tree

Compares the three models trained with `src/train.py`. Each one was tuned on the 1M sample using val macro-F1, then the best setting was refit on all of train and scored once on val and test.

To make the results files again:

```
python -m src.train --model lr
python -m src.train --model nb
python -m src.train --model dt
python -m src.train --model dt --extra year Source
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # so `src` imports from the notebooks folder
from src import config
from src.models import fit_model, make_model

config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# same style as notebooks 01 and 02
INK = '#52514e'
GRAY = '#8a8980'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK, 'text.color': '#1a1a19',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10,
})

# one color per model, same everywhere
MODELS = {
    'lr': ('Logistic Regression', '#2a78d6'),
    'nb': ('Naive Bayes', '#eb6834'),
    'dt': ('Decision Tree', '#1baf7a'),
    'dt_plus_year_source': ('Decision Tree + year/Source', '#eda100'),
}
MAIN = ['lr', 'nb', 'dt']

## Results

The baseline always guesses severity 2, the most common class.

In [ ]:
def load(name):
    with open(config.RESULTS_DIR / f'{name}.json') as f:
        return json.load(f)

results = {m: {split: load(f'{m}_{split}') for split in ['val', 'test']} for m in MODELS}

# baseline from the test class counts: always guess 2
support = {c: results['dt']['test']['per_class'][str(c)]['support'] for c in config.CLASSES}
share_2 = support[2] / sum(support.values())
baseline = {'accuracy': share_2, 'macro_f1': (2 * share_2 / (1 + share_2)) / 4}

rows = []
for m, (label, _) in MODELS.items():
    rows.append({
        'model': label,
        'val macro-F1': results[m]['val']['macro_f1'],
        'test macro-F1': results[m]['test']['macro_f1'],
        'test accuracy': results[m]['test']['accuracy'],
    })
rows.append({'model': 'Always guess 2', 'val macro-F1': np.nan,
             'test macro-F1': baseline['macro_f1'], 'test accuracy': baseline['accuracy']})
summary = pd.DataFrame(rows).set_index('model')
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
names = list(MODELS)
scores = [results[m]['test']['macro_f1'] for m in names]
ax.barh([MODELS[m][0] for m in names], scores, color=[MODELS[m][1] for m in names], height=0.6)
ax.axvline(baseline['macro_f1'], color=GRAY, linestyle='--', linewidth=1.5)
ax.text(baseline['macro_f1'] + 0.005, 0.5, 'always guess 2', va='center', color=INK, fontsize=9)
for i, s in enumerate(scores):
    ax.text(s + 0.005, i, f'{s:.3f}', va='center', color=INK)
ax.invert_yaxis()
ax.set_xlim(0, 0.6)
ax.set_xlabel('test macro-F1')
ax.set_title('Test macro-F1 by model')
ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'model_macro_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## F1 for each severity

Macro-F1 is the average of these four, so a model only scores well if it finds the rare severities too.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.2
x = np.arange(len(config.CLASSES))
for i, m in enumerate(MODELS):
    f1 = [results[m]['test']['per_class'][str(c)]['f1'] for c in config.CLASSES]
    ax.bar(x + (i - 1.5) * width, f1, width - 0.02, color=MODELS[m][1], label=MODELS[m][0])
ax.set_xticks(x, [f'Severity {c}' for c in config.CLASSES])
ax.set_ylabel('test F1')
ax.set_ylim(0, 1)
ax.set_title('Test F1 by severity')
ax.legend(frameon=False, fontsize=9)
ax.grid(axis='x', visible=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'model_f1_by_severity.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion matrices

Each row is the true severity, split by what the model guessed (row %). A good model has dark squares on the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, m in zip(axes, MAIN):
    cm = np.array(results[m]['test']['confusion_matrix'])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ax.imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{pct[i, j]:.0f}', ha='center', va='center',
                    color='white' if pct[i, j] > 50 else '#1a1a19')
    ax.set_xticks(range(4), config.CLASSES)
    ax.set_yticks(range(4), config.CLASSES)
    ax.set_xlabel('predicted')
    ax.set_title(MODELS[m][0])
    ax.grid(False)
axes[0].set_ylabel('true')
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'model_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## Twin check

The split is random, so some test rows have the exact same feature values as a row in train (same time, same weather report, same road). A deep tree can just remember those. This scores the best tree and naive bayes again, split into test rows with and without a twin in train.

This refits both models on all of train, so it takes a few minutes.

In [ ]:
features = config.MODEL_FEATURES
train = pd.read_parquet(config.TRAIN_PARQUET, columns=features + [config.TARGET])
test = pd.read_parquet(config.TEST_PARQUET, columns=features + [config.TARGET])

# same features as a row in train
def row_key(df):
    return pd.util.hash_pandas_object(df[features].astype(str), index=False)

twin = row_key(test).isin(set(row_key(train))).values
print(f'test rows with a twin in train: {twin.mean():.1%}')

In [ ]:
from sklearn.metrics import f1_score

best = {
    'nb': make_model('nb', features, model__alpha=0.01),
    'dt': make_model('dt', features, model__max_depth=None, model__min_samples_leaf=1),
}
y = test[config.TARGET].values
twin_rows = []
for m, pipe in best.items():
    pred = fit_model(pipe, train[features], train[config.TARGET]).predict(test[features])
    for group, mask in [('has twin', twin), ('no twin', ~twin)]:
        twin_rows.append({'model': MODELS[m][0], 'test rows': group,
                          'macro-F1': f1_score(y[mask], pred[mask], labels=config.CLASSES,
                                               average='macro', zero_division=0)})
twin_table = pd.DataFrame(twin_rows).pivot(index='model', columns='test rows', values='macro-F1')
twin_table.round(3)

## Observations

- Decision tree is the best of the three (test macro-F1 0.414), then naive bayes (0.328), then logistic regression (0.238). All three beat always guessing 2 (0.220).
- None of them beat the 0.787 accuracy you get from always guessing 2, so accuracy isn't a good way to compare models here. Macro-F1 is.
- Logistic regression barely beats the baseline. It treats hour, month and weekday as straight-line numbers, so it can't pick up things like rush hour peaks.
- GaussianNB scored below the baseline because most features are 0/1, so we switched to CategoricalNB with binned weather numbers.
- Severity 4 is the hardest. Only the decision tree finds some of it (F1 0.24), naive bayes almost never predicts it.
- Part of the tree's score is memorizing. 28% of test rows have the same features as a train row, and the tree scores 0.601 on those but 0.343 on the rest. Naive bayes is about the same on both (0.339 vs 0.322), so the real gap between them is small.
- Adding year and Source pushes the tree to 0.505, but that's from how each source labels severity, not the crash itself. Severity 4 actually gets worse (0.24 to 0.12), so they stay out.